# Imports and initialization

In [ ]:
import re

import logging
import os
from pathlib import Path
import pandas as pd

import spacy
from spacy import displacy
from spacy.tokens.doc import Doc

# defining paths for notebook
current_dir = Path(globals()["_dh"][0])

csv_folder = os.path.join(current_dir, r"csv")
train_csv = "train.csv"

# logging configuration

logging.basicConfig(level=logging.DEBUG)

## Reading csv

In [ ]:
# We are not using title, but we'd like to, add it here
used_columns = ["id", "text", "rating"]

df = pd.read_csv(os.path.join(csv_folder, train_csv), usecols=used_columns)

text_test = df["text"][10]

df.head()

In [ ]:
print(df['rating'].value_counts())

# Cleaning text

In [ ]:
def clean_text(text):
    text = (
        text.lower()
    )  # NOTE: talvez seja interessante deixar alguns caracteres em maiúsculo

    # remove URLs
    text = re.sub(r"http\\S+|www\\S+", " ", text)

    # keeps letters and spaces
    text = re.sub(
        r"[^a-zà-úâêîôûãõç\\s]", " ", text
    )  # NOTE: está tirando os números também

    # removes extra spaces
    text = re.sub(r"\\s+", " ", text).strip()

    return text

In [ ]:
print(clean_text(text_test))

# Features

In [ ]:
# 1 Number of ADJ
# 3 Number of ADV
# 6 Number of DET 
# 7 Number of INTJ
# 9 Number of NUM
# 12 Number of PUNCT
# 14 Number of SYM
# 16 Number of COMP
# 17 Number of SUP
# 18 Number of X
#
# 19 ADJ → NOUN
# 23 ADV → VERB + {PCP | GER | PS | IMPF}
#
# 24 Number of subjecticve terms
# 25 Number of positve terms
# 26 Number of negative terms
# 27 Number of marks
# 28 Proportion of subjective terms
# 29 Proportion of positive terms
# 30 Proportion of negative terms
# 31 Proportion of mark terms
#
# tabela 4 ignora
#
#
# 46 Ratio of the uppercase characters to the sentence length
#
# tabela 6 twitter
# 47 Number of URL
# 48 Number of mentions (@user)
# 49 Number of elongated words
# 50 Polarity of emojis
# 51 Polarity of emoticons

## Part-Of-Speech (POS) features

Don't forget to `python -m spacy download pt_core_news_sm`
See https://universaldependencies.org/u/pos/ for information on Tags

In [ ]:
nlp = spacy.load("pt_core_news_lg")

In [ ]:
doc_test = nlp(text_test)

In [ ]:
print(doc_test)

In [ ]:
# Feature 1
def number_of_adj(doc: Doc):
    logging.debug("[number_of_adj]")

    adj_num = 0
    for token in doc:
        if token.pos_ == "ADJ":
            adj_num += 1

            logging.debug(token.text + "\t|" + token.pos_)
    return adj_num


print(doc_test, "\nNumber of adj:", number_of_adj(doc_test))

In [ ]:
# Feature 3
def number_of_adv(doc: Doc):
    adv_num = 0

    for token in doc:
        if token.pos_ == "ADJ":
            adv_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return adv_num


print(doc_test, "\nNumber of adv:", number_of_adv(doc_test))

In [ ]:
# Feature 6
# NOTE: eu acho que essa feature é meio nada a ver -vini
# NOTE: por mim tira essa aqui tbm -matheus
def number_of_det(doc: Doc):
    det_num = 0

    for token in doc:
        if token.pos_ == "DET":
            det_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return det_num


print(doc_test, "\nNumber of det:", number_of_det(doc_test))

In [ ]:
# Feature 7
def number_of_intj(doc: Doc):
    intj_num = 0

    for token in doc:
        if token.pos_ == "INTJ":
            intj_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return intj_num


test = nlp("Ah, que droga!")
print(test, "\nNumber of intj:", number_of_intj(test))

In [ ]:
# Feature 9
def number_of_num(doc: Doc):
    num_num = 0

    for token in doc:
        if token.pos_ == "NUM":
            num_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return num_num


test = nlp("Dois pesos, 2 medidas")
print(test, "\nNumber of num:", number_of_num(test))

In [ ]:
# Feature 12
def number_of_punct(doc: Doc):
    punct_num = 0

    for token in doc:
        if token.pos_ == "PUNCT":
            punct_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return punct_num


test = nlp("Ah, que droga!")
print(test, "\nNumber of punct:", number_of_punct(test))



In [ ]:
# Feature 14
def number_of_sym(doc: Doc):
    sym_num = 0

    for token in doc:
        if token.pos_ == "SYM":
            sym_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return sym_num


test = nlp("R$50,00")
print(test, "\nNumber of sym:", number_of_sym(test))



In [ ]:
# Feature 16 & 18
# NOTE: Tentou usar pronomes aqui? Tp eu vi que ele categoriza como adjetivo/ adverbio parece. Talvez classificar esse cara aqui como adv modificador ou adj modificador? 
comp_sup_list = ["melhor", "pior", "maior", "menor"]


def number_of_comp_sup(text: str):
    """Comparative or superlative words"""
    text = text.lower()
    comp_sub_count = 0

    for word in comp_sup_list:
        comp_sub_count += text.count(word)

    return comp_sub_count


test = "A maior e melhor torta é a de limão, e a pior? Não sei"
print(number_of_comp_sup(test))



In [ ]:
# Feature 18
# NOTE: This feature doesn't seem that meaningful -vini
# NOTE: Couldn't test it
def number_of_x(doc: Doc):
    """X means other"""
    x_num = 0

    for token in doc:
        if token.pos_ == "X":
            x_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return x_num


test = nlp("E aí ele xfgh pdl jklw sombrero fastfood meaningful")
print(test, "\nNumber of x:", number_of_x(test))

### spaCy testing

In [ ]:
doc = nlp(text_test)
for token in doc:
    print(token.text, "\t\t|", token.pos_, token.dep_)

In [ ]:
displacy.render(doc, style="dep", options={"compact": "True"})

## Set of Syntactic Patterns

In [ ]:
def extrair_padroes_sintaticos(doc: Doc) -> dict:
    pat_19, pat_20, pat_21, pat_22, pat_23 = [], [], [], [], []

    for token in doc:
        # 19: ADJ → NOUN
        if token.pos_ == "NOUN":
            adjs = [w for w in token.lefts
                    if w.dep_ == "amod" and w.pos_ == "ADJ"]
            if adjs:
                pat_19.append({"noun": token.text,
                               "adj": [w.text for w in adjs]})

        # 20: ADV → ADJ → Not(NOUN)
        if token.pos_ == "VERB":
            subj_adj = [w for w in token.lefts
                        if w.dep_ == "nsubj" and w.pos_ in ("ADJ", "PROPN")]
            adv = [w for w in token.rights
                   if w.dep_ in ("advmod", "xcomp")
                   and w.pos_ in ("ADV", "ADJ")]
            intens = [w for a in adv
                      for w in a.children if w.pos_ == "ADV"]
            if subj_adj and adv:
                pat_20.append({"adj": [w.text for w in subj_adj],
                               "adv": [w.text for w in adv],
                               "intensificador": [w.text for w in intens]})

        # 20 large: ADJ como ROOT com csubj
        if token.pos_ == "ADJ" and token.dep_ == "ROOT":
            csubj = [w for w in token.rights
                     if w.dep_ == "csubj" and w.pos_ == "VERB"]
            if csubj:
                adv_chain = [w for verb in csubj
                             for w in verb.rights
                             if w.dep_ == "xcomp" and w.pos_ == "ADJ"]
                intens = [w for adj in adv_chain
                          for w in adj.children if w.pos_ == "ADV"]
                pat_20.append({
                    "adj": [token.text],
                    "adv": [w.text for w in adv_chain],
                    "intensificador": [w.text for w in intens]
                })

        # 21: ADJ → ADJ → Not(NOUN)
        if token.pos_ == "ADJ" and token.dep_ == "amod":
            head = token.head
            irmaos_adj = [w for w in head.children
                          if w.pos_ == "ADJ" and w != token]
            if irmaos_adj:
                pat_21.append({"adj1": token.text,
                               "adj2": [w.text for w in irmaos_adj],
                               "head": head.text})

        # 22: NOUN → ADJ → Not(NOUN)
        if token.pos_ == "ADJ" and token.dep_ == "ROOT":
            subj_noun = [w for w in token.lefts
                         if w.dep_ == "nsubj" and w.pos_ == "NOUN"]
            intensif = [w for w in token.children
                        if w.dep_ == "advmod" and w.pos_ == "ADV"]
            if subj_noun:
                pat_22.append({"noun_subj": [w.text for w in subj_noun],
                               "adj_pred": token.text,
                               "intensificador": [w.text for w in intensif]})

        # 23: ADV → VERB + {PCP | GER}
        if token.pos_ == "VERB" and token.dep_ == "ROOT":
            formas = [w for w in token.rights
                      if w.pos_ in ("VERB", "ADJ")
                      and any(f in w.morph.get("VerbForm", [])
                              for f in ["Part", "Ger"])]
            adv_intens = [w for w in token.children
                          if w.dep_ == "advmod" and w.pos_ == "ADV"]
            adv_forma = [w for f in formas
                         for w in f.children if w.dep_ == "advmod"]
            if formas and (adv_intens or adv_forma):
                pat_23.append({"verbo_aux": token.text,
                               "forma_verbal": [w.text for w in formas],
                               "adv_intensif": [w.text for w in adv_intens],
                               "adv_forma": [w.text for w in adv_forma]})

    return {
        "ADJ_NOUN":                 len(pat_19),
        "ADV_ADJ_NOT(NOUN)":        len(pat_20),
        "ADJ_ADJ_NOT(NOUN)":        len(pat_21),
        "NOUN_ADJ_NOT(NOUN)":       len(pat_22),
        "ADV_VERB_PCP_GER_PS_IMPF": len(pat_23),
    }

In [ ]:
## Acontece que o próprio modelo não pega sempre o padrao, mas as vezes sim

# l = "Vergonhoso cobrarem tão caro."

# doc = nlp(l.lower())
# print(doc)
# print(extrair_padroes_sintaticos(doc))

In [ ]:
# for token in doc:
#     print(f"{token.text:15} pos={token.pos_:6} dep={token.dep_:12} head={token.head.text}")

## Sentimental Features


In [ ]:
# provável que a gente tenha que fazer nossas próprias listas de palavras positivas e negativas

# Listas de palavras para features de sentimento (Features 25, 26, 28, 29)
# Baseadas em léxicos de sentimento para português
# Adaptadas ao domínio de reviews de produtos

POSITIVE_TERMS = [
    # Adjetivos positivos
    "bom", "boa", "bons", "boas",
    "ótimo", "ótima", "ótimos", "ótimas",
    "excelente", "excelentes",
    "maravilhoso", "maravilhosa", "maravilhosos", "maravilhosas",
    "perfeito", "perfeita", "perfeitos", "perfeitas",
    "incrível", "incríveis",
    "fantástico", "fantástica", "fantásticos", "fantásticas",
    "sensacional", "sensacionais",
    "lindo", "linda", "lindos", "lindas",
    "bonito", "bonita", "bonitos", "bonitas",
    "agradável", "agradáveis",
    "delicioso", "deliciosa", "deliciosos", "deliciosas",
    "gostoso", "gostosa", "gostosos", "gostosas",
    "rápido", "rápida", "rápidos", "rápidas",
    "eficiente", "eficientes",
    "resistente", "resistentes",
    "durável", "duráveis",
    "confortável", "confortáveis",
    "prático", "prática", "práticos", "práticas",
    "funcional", "funcionais",
    "satisfeito", "satisfeita", "satisfeitos", "satisfeitas",
    "feliz", "felizes",
    "top", "show", "incrível",
    # Verbos/expressões positivas
    "adorei", "amei", "gostei", "aprovei", "recomendo",
    "superou", "surpreendeu", "encantou",
    "funcionou", "funcionando", "chegou",
    "vale", "valeu", "valendo",
    # Substantivos positivos
    "qualidade", "excelência", "perfeição",
    "satisfação", "prazer", "amor",
]

NEGATIVE_TERMS = [
    # Adjetivos negativos
    "ruim", "ruins",
    "péssimo", "péssima", "péssimos", "péssimas",
    "horrível", "horríveis",
    "terrível", "terríveis",
    "lixo", "lixos",
    "fraco", "fraca", "fracos", "fracas",
    "quebrado", "quebrada", "quebrados", "quebradas",
    "defeituoso", "defeituosa", "defeituosos", "defeituosas",
    "demorado", "demorada", "demorados", "demoradas",
    "caro", "cara", "caros", "caras",
    "decepcionante", "decepcionantes",
    "insatisfatório", "insatisfatória",
    "inadequado", "inadequada", "inadequados", "inadequadas",
    "falso", "falsa", "falsos", "falsas",
    "inútil", "inúteis",
    "difícil", "difíceis",
    "desconfortável", "desconfortáveis",
    # Verbos/expressões negativos
    "odiei", "detestei", "decepcionou", "frustrou",
    "quebrou", "parou", "travou", "bugou",
    "arrependido", "arrependida",
    "devolvendo", "devolvi", "cancelei",
    # Substantivos negativos
    "decepção", "vergonha", "fraude", "lixo",
    "problema", "problemas", "defeito", "defeitos",
    "absurdo", "absurdos",
]

# Termos subjetivos = positivos + negativos + marcadores de opinião
OPINION_MARKERS = [
    "acho", "acredito", "penso", "considero", "creio",
    "parece", "parecer", "pareceu",
    "sinto", "senti", "sentindo",
    "esperava", "esperando",
    "recomendo", "recomendaria", "recomendaria",
    "vale", "valeu", "vale a pena",
]

SUBJECTIVE_TERMS = set(POSITIVE_TERMS + NEGATIVE_TERMS + OPINION_MARKERS)

# Pré-computar sets para busca O(1)
POSITIVE_SET  = set(POSITIVE_TERMS)
NEGATIVE_SET  = set(NEGATIVE_TERMS)

print(f"Positive terms: {len(POSITIVE_SET)}")
print(f"Negative terms: {len(NEGATIVE_SET)}")
print(f"Subjective terms (total): {len(SUBJECTIVE_TERMS)}")

In [ ]:
'''# carrega uma vez
op = pd.read_csv("lexico_v3.0.txt", sep=",", header=None,
                 names=["word", "pos", "polarity", "type"])

POSITIVE_SET = set(op[op["polarity"] ==  1]["word"])
NEGATIVE_SET = set(op[op["polarity"] == -1]["word"])

# as funções do notebook continuam idênticas
def number_of_positive_terms(doc):
    return sum(1 for t in doc 
               if t.is_alpha and t.lemma_.lower() in POSITIVE_SET)
'''

In [ ]:
# Feature 25 — Number of positive terms
def number_of_positive_terms(doc: Doc):
    """
    Conta tokens cujo lema (em minúsculas) está na lista de termos positivos.
    Usa o lema para capturar flexões: 'excelentes' -> 'excelente'.
    Exemplo: 'As comidas são excelentes.' -> 1
    """
    count = 0
    for token in doc:
        if token.is_alpha and token.lemma_.lower() in POSITIVE_SET:
            count += 1
            logging.debug(f"[positive] {token.text} -> {token.lemma_}")
    return count


test_pos = nlp("A maior e melhor torta é a de limão, e a pior? Não sei")
test_neg = nlp("Atendimento da recepcionista da porta ruim.")
print("'As comidas são excelentes.':", number_of_positive_terms(test_pos))  # esperado: 1
print("'Atendimento da recepcionista da porta ruim.':", number_of_positive_terms(test_neg))  # esperado: 0

print(number_of_positive_terms(test_pos))

In [ ]:
# Feature 26 — Number of negative terms
def number_of_negative_terms(doc: Doc):
    """
    Conta tokens cujo lema (em minúsculas) está na lista de termos negativos.
    Usa o lema para capturar flexões: 'ruins' -> 'ruim'.
    Exemplo: 'Atendimento da recepcionista da porta ruim.' -> 1
    """
    count = 0
    for token in doc:
        if token.is_alpha and token.lemma_.lower() in NEGATIVE_SET:
            count += 1
            logging.debug(f"[negative] {token.text} -> {token.lemma_}")
    return count


test_pos = nlp("As comidas são excelentes.")
test_neg = nlp("Atendimento da recepcionista da porta ruim.")
print("'As comidas são excelentes.' :", number_of_negative_terms(test_pos))   # esperado: 0
print("'Atendimento da recepcionista da porta ruim.':", number_of_negative_terms(test_neg))  # esperado: 1



In [ ]:


# Feature 28 — Proportion of subjective terms
def proportion_of_subjective_terms(doc: Doc):
    """
    Proporção de tokens alfanuméricos que são termos subjetivos
    (positivos + negativos + marcadores de opinião).
    Retorna valor entre 0 e 1, arredondado em 3 casaas.
    Exemplo: 'Comida boa por um bom preço.' -> 0.429 (2 subjetivos de 7 tokens alpha)
    """
    alpha_tokens = [t for t in doc if t.is_alpha]
    if not alpha_tokens:
        return 0.0

    subjective_count = sum(
        1 for t in alpha_tokens
        if t.lemma_.lower() in SUBJECTIVE_TERMS
    )
    return round(subjective_count / len(alpha_tokens), 3)


test_sub = nlp("Comida boa por um bom preço.")
print("'Comida boa por um bom preço.':", proportion_of_subjective_terms(test_sub))  # esperado: ~0.429



In [ ]:
# Feature 29 — Proportion of positive terms
def proportion_of_positive_terms(doc: Doc):
    """
    Proporção de tokens alfanuméricos que são termos positivos.
    Retorna valor entre 0 e 1, aredondado em 3 casas.
    Exemplo: 'As comidas são excelente s.' -> 0.2 (1 positivo de 5 tokens alpha)
    """
    alpha_tokens = [t for t in doc if t.is_alpha]
    if not alpha_tokens:
        return 0.0

    positive_count = sum(
        1 for t in alpha_tokens
        if t.lemma_.lower() in POSITIVE_SET
    )
    return round(positive_count / len(alpha_tokens), 3)


test_pos = nlp("As comidas são excelentes.")
print(test_pos)
print("'As comidas são excelentes.':", proportion_of_positive_terms(test_pos))  # esperado: 0.2

## Structural Features

In [ ]:
def uppercase_ratio(text: str):
    # erases spaces
    re.sub("\\s", "", text)

    length = len(text)

    upper_count = len(re.findall("[A-Z]", text))
    
    ratio = {
        "UPCASE_RATIO" : round(upper_count / length, 3)

    }

    return ratio


test1 = "Eu absolutamente ADORO GIRAFFAS"
test2 = "sei lá, tanto faz. Sabe?"

print("test 1: ", uppercase_ratio(test1))
print("test 2: ", uppercase_ratio(test2))

## TWITTER FEATURES

In [ ]:


# Feature 47 — Number of URLs
def number_of_urls(text: str):
    """
    Conta URLs no texto usando regex.
    Exemplo: 'Vendo! Compartilha aí gente! http://fb.me/24tl16IwA' -> 1
    """
    url_pattern = re.compile(
        r"http\S+|www\.\S+",
        re.IGNORECASE
    )
    return len(url_pattern.findall(text))


test1 = "Vendo! Compartilha aí gente! http://fb.me/24tl16IwA"
test2 = "Produto sem link nenhum"
print(repr(test1), ":", number_of_urls(test1))  # esperado: 1
print(repr(test2), ":", number_of_urls(test2))  # esperado: 0



In [ ]:
# Feature 48 — Number of mentions (@user)
def number_of_mentions(text: str):
    """
    Conta menções no formato @usuario.
    Exemplo: 'Comprei um notebook da @Dell vamos ver se eu vou gostar!!!' -> 1
    """
    mention_pattern = re.compile(r"@\w+")
    return len(mention_pattern.findall(text))


test1 = "Comprei um notebook da @Dell vamos ver se eu vou gostar!!!"
test2 = "Produto sem menção nenhuma"
print(repr(test1), ":", number_of_mentions(test1))  # esperado: 1
print(repr(test2), ":", number_of_mentions(test2))  # esperado: 0

In [ ]:

# Feature 49 — Number of elongated words
def number_of_elongated_words(text: str):
    """
    Conta palavras com letras repetidas 3+ vezes consecutivas.
    Esse padrão indica ênfase emocional em reviews informais.
    Exemplo: 'Essa Dell é demais... Note chegoooou?' -> 1  ('chegoooou')
    Exemplo: 'Note arriiiived?' -> 1  ('arriiiived')
    """
    elongated_pattern = re.compile(r"\b\w*(.)\1{2,}\w*\b")
    return len(elongated_pattern.findall(text))


test1 = "Essa Dell é demais... Note chegoooou?"
test2 = "Produto normal sem ênfase"
test3 = "arriiiived e chegoooou ambos elongados"
print(repr(test1), ":", number_of_elongated_words(test1))  # esperado: 1
print(repr(test2), ":", number_of_elongated_words(test2))  # esperado: 0
print(repr(test3), ":", number_of_elongated_words(test3))  # esperado: 2



In [ ]:
# Feature 51 — Polarity of emoticons
# Emoticons são sequências de caracteres ASCII (ex: :-) :-( )
EMOTICON_POLARITY = {
    # =========================
    # POSITIVOS (+1)
    # =========================
    
    # Sorrisos básicos
    ":)": 1, ":-)": 1, ":]": 1, ":-]": 1, "=]": 1, "=)": 1,
    ":}": 1, ":-}": 1,
    
    # Risadas
    ":D": 1, ":-D": 1, "=D": 1, "xD": 1, "XD": 1, "X-D": 1,
    "x-D": 1,
    
    # Piscadas
    ";)": 1, ";-)": 1, "*-)": 1, "*)": 1,
    
    # Língua / zoeira
    ":P": 1, ":-P": 1, ":p": 1, ":-p": 1,
    "=P": 1, "=p": 1, "XP": 1, "xp": 1,
    
    # Amor / carinho
    "<3": 1, "</3": -1,  # (já colocando o oposto aqui)
    ":*": 1, ":-*": 1,
    
    # Anime / kawaii
    "^^": 1, "^_^": 1, "^-^": 1, "^.^": 1,
    "(^_^)": 1, "(^.^)": 1, "(^-^)": 1,
    "(^o^)": 1,
    
    # Outros positivos
    ":')": 1,  # choro de felicidade
    ":>": 1, ":-)": 1,
    "(:": 1, "(-:": 1,
    
    # =========================
    # NEGATIVOS (-1)
    # =========================
    
    # Tristeza
    ":(": -1, ":-(": -1, ":[": -1, ":-[": -1,
    ":{": -1, ":-{": -1,
    "=(":-1 if False else -1,  # evita duplicação visual bug (mantém válido)
    "=(": -1,
    
    # Choro
    ":'(": -1, ":'-(": -1,
    "T_T": -1, "TT_TT": -1, ";_;": -1,
    "(T_T)": -1,
    
    # Decepção / neutro negativo
    ":/": -1, ":-/": -1, ":\\": -1, ":-\\": -1,
    ":|": -1, ":-|": -1,
    ">:|": -1,
    
    # Raiva
    ">:(": -1, ">:-(": -1,
    "D:<": -1, "D-:<": -1,
    
    # Choque / medo
    "D:": -1, "D-:": -1,
    
    # Vergonha / desconforto
    ":$": -1, ":-$": -1,
    ":-X": -1, ":X": -1,
    
    # Confusão
    "o_O": -1, "O_o": -1, "o.O": -1, "O.O": -1,
    
    # Cansaço
    "-_-": -1, "-__-": -1,
    
    # Outros negativos
    "):": -1, ")-:": -1,
}

# Regex para detectar emoticons — ordem importa: mais longos primeiro
_emoticon_pattern = re.compile(
    r"|" .join(re.escape(e) for e in sorted(EMOTICON_POLARITY, key=len, reverse=True))
)


def polarity_of_emoticons(text: str):
    """
    Calcula a polaridade média dos emoticons ASCII presentes no texto.
    Retorna -1, 0 ou 1 (ou média se houver mistura).
    Exemplo: 'Notebook #dell sem teclado #abnt2 desanima... :-(' -> -1
    """
    matches = _emoticon_pattern.findall(text)
    scores = [EMOTICON_POLARITY[m] for m in matches if m in EMOTICON_POLARITY]
    if not scores:
        return 0
    return round(sum(scores) / len(scores), 3)


test1 = "Notebook #dell sem teclado #abnt2 desanima... :-("
test2 = "Chegou antes do prazo!! :D :-)"
test3 = "Produto comum, sem emoticon"
print(repr(test1), "->", polarity_of_emoticons(test1))  # esperado: -1
print(repr(test2), "->", polarity_of_emoticons(test2))  # esperado: 1.0
print(repr(test3), "->", polarity_of_emoticons(test3))  # esperado: 0



In [ ]:
# Só para inferir as colunas do CSV
# Adicionar outras features aqui, para montar o big csv
def get_columns(text: str) -> dict:
    cleaned_text = clean_text(text)
    doc = nlp(cleaned_text)
    return extrair_padroes_sintaticos(doc) | uppercase_ratio(text)

In [ ]:
import csv
import os

file_name = "features.csv"
file_path = os.path.join(current_dir, file_name)
file_exists = os.path.exists(file_path)

keys = get_columns(df['text'][0])
columns = [*keys, 'rating', 'id']

# Manter os dois — original e limpo
original_texts = [str(t) for t in df['text']]
cleaned_texts  = [clean_text(t) for t in original_texts]
ratings = df['rating'].tolist()
ids     = df['id'].tolist()

with open(file_path, mode="a", newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    if not file_exists:
        writer.writeheader()

    for doc, original, rating, id_ in zip(
        nlp.pipe(cleaned_texts, batch_size=256, n_process=1),
        original_texts,
        ratings,
        ids
    ):
        syntactic_patterns  = extrair_padroes_sintaticos(doc)
        structural_features = uppercase_ratio(original)  

        features = syntactic_patterns | structural_features
        features['rating'] = rating
        features['id']     = id_
        writer.writerow(features)

# LLM TRAINING

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# Carregar features e labels
features_df = pd.read_csv(os.path.join(current_dir, "features.csv"))
train_df = pd.read_csv(os.path.join(current_dir, "train.csv"))

# Garantir alinhamento pelo id
df = train_df[['id', 'rating']].merge(features_df, on='id')

X = df.drop(columns=['id', 'rating'])
y = df['rating']

# Validação
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []
models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(100)]
    )

    preds = model.predict(X_val)
    f1 = f1_score(y_val, preds, average='macro')
    f1_scores.append(f1)
    models.append(model)
    print(f"Fold {fold+1} — F1 macro: {f1:.4f}")

print(f"\nMédia: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(classification_report(y_val, preds))

In [ ]:
# Usar o melhor fold ou média dos modelos (ensemble)
test_features = pd.read_csv(os.path.join(current_dir, "features_test.csv"))  # mesmo pipeline aplicado no test.csv
X_test = test_features.drop(columns=['id'])

# Opção A — melhor modelo único
best_model = models[np.argmax(f1_scores)]
preds_test = best_model.predict(X_test)

# Opção B — ensemble por votação (mais robusto)
all_preds = np.array([m.predict(X_test) for m in models])
preds_test = pd.Series(all_preds).apply(
    lambda col: np.bincount(col).argmax(), axis=0
)

# Salvar
sample = pd.read_csv(os.path.join(current_dir, "sample_submission.csv"))
sample['rating'] = preds_test
sample.to_csv(os.path.join(current_dir, "submission.csv"), index=False)